# 05 — Data Analysis: 20 Preguntas de Negocio

Todas las consultas se ejecutan contra `NYC_TAXI_P3.ANALYTICS.OBT_TRIPS` usando Spark + Snowflake pushdown.

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("NYC_Taxi_Analysis").getOrCreate()

SF_OPTIONS = {
    "sfURL":       f"{os.environ['SF_ACCOUNT']}.snowflakecomputing.com",
    "sfUser":      os.environ["SF_USER"],
    "sfPassword":  os.environ["SF_PASSWORD"],
    "sfDatabase":  os.environ["SF_DATABASE"],
    "sfSchema":    os.environ["SF_SCHEMA_RAW"],
    "sfWarehouse": os.environ["SF_WAREHOUSE"],
    "sfRole":      os.environ["SF_ROLE"],
}
SF_OPTS_ANALYTICS = {**SF_OPTIONS, "sfSchema": os.environ["SF_SCHEMA_ANALYTICS"]}

def query_obt(sql):
    """Ejecuta SQL pushdown contra OBT_TRIPS y retorna Spark DataFrame."""
    return spark.read.format("snowflake").options(**SF_OPTS_ANALYTICS).option("query", sql).load()

print("Listo para análisis")

---
## (a) Top 10 zonas de pickup por volumen mensual

In [ ]:
qa = """
SELECT pu_zone, pu_borough,
       COUNT(*) AS total_trips,
       ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT (year * 100 + month)), 0), 0) AS avg_monthly_trips
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
WHERE pu_zone IS NOT NULL
GROUP BY pu_zone, pu_borough
ORDER BY total_trips DESC
LIMIT 10
"""
query_obt(qa).show(truncate=False)

## (b) Top 10 zonas de dropoff por volumen mensual

In [ ]:
qb = """
SELECT do_zone, do_borough,
       COUNT(*) AS total_trips,
       ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT (year * 100 + month)), 0), 0) AS avg_monthly_trips
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
WHERE do_zone IS NOT NULL
GROUP BY do_zone, do_borough
ORDER BY total_trips DESC
LIMIT 10
"""
query_obt(qb).show(truncate=False)

## (c) Evolución mensual de total_amount y tip_pct por borough

In [ ]:
qc = """
SELECT pu_borough, year, month,
       ROUND(AVG(total_amount), 2)  AS avg_total_amount,
       ROUND(AVG(tip_pct), 2)       AS avg_tip_pct,
       COUNT(*)                     AS trips
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
WHERE pu_borough IS NOT NULL
GROUP BY pu_borough, year, month
ORDER BY pu_borough, year, month
"""
query_obt(qc).show(100, truncate=False)

## (d) Ticket promedio (avg total_amount) por service_type y mes

In [ ]:
qd = """
SELECT service_type, year, month,
       ROUND(AVG(total_amount), 2) AS avg_ticket,
       COUNT(*)                    AS trips
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
GROUP BY service_type, year, month
ORDER BY service_type, year, month
"""
query_obt(qd).show(100, truncate=False)

## (e) Viajes por hora del día y día de semana (picos)

In [ ]:
qe = """
SELECT day_of_week, pickup_hour,
       COUNT(*) AS trips
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
GROUP BY day_of_week, pickup_hour
ORDER BY day_of_week, pickup_hour
"""
query_obt(qe).show(200, truncate=False)

## (f) p50 / p90 de trip_duration_min por borough de pickup

In [ ]:
qf = """
SELECT pu_borough,
       ROUND(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY trip_duration_min), 2) AS p50_duration,
       ROUND(PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY trip_duration_min), 2) AS p90_duration,
       COUNT(*) AS trips
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
WHERE pu_borough IS NOT NULL AND trip_duration_min IS NOT NULL AND trip_duration_min >= 0
GROUP BY pu_borough
ORDER BY p50_duration DESC
"""
query_obt(qf).show(truncate=False)

## (g) avg_speed_mph por franja horaria (6–9, 17–20) y borough

In [ ]:
qg = """
SELECT pu_borough,
       CASE
           WHEN pickup_hour BETWEEN 6 AND 9   THEN 'Morning Rush (6-9)'
           WHEN pickup_hour BETWEEN 17 AND 20  THEN 'Evening Rush (17-20)'
           ELSE 'Other Hours'
       END AS time_slot,
       ROUND(AVG(avg_speed_mph), 2) AS avg_speed,
       COUNT(*) AS trips
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
WHERE pu_borough IS NOT NULL AND avg_speed_mph IS NOT NULL
GROUP BY pu_borough, time_slot
ORDER BY pu_borough, time_slot
"""
query_obt(qg).show(30, truncate=False)

## (h) Participación por payment_type_desc y su relación con tip_pct

In [ ]:
qh = """
SELECT payment_type_desc,
       COUNT(*) AS trips,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_total,
       ROUND(AVG(tip_pct), 2) AS avg_tip_pct,
       ROUND(AVG(tip_amount), 2) AS avg_tip_amount
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
GROUP BY payment_type_desc
ORDER BY trips DESC
"""
query_obt(qh).show(truncate=False)

## (i) ¿Qué rate_code_desc concentran mayor trip_distance y total_amount?

In [ ]:
qi = """
SELECT rate_code_desc,
       COUNT(*) AS trips,
       ROUND(AVG(trip_distance), 2) AS avg_distance,
       ROUND(SUM(trip_distance), 0) AS total_distance,
       ROUND(AVG(total_amount), 2)  AS avg_total_amount,
       ROUND(SUM(total_amount), 0)  AS sum_total_amount
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
WHERE rate_code_desc IS NOT NULL
GROUP BY rate_code_desc
ORDER BY sum_total_amount DESC
"""
query_obt(qi).show(truncate=False)

## (j) Mix yellow vs green por mes y borough

In [ ]:
qj = """
SELECT pu_borough, year, month, service_type,
       COUNT(*) AS trips,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY pu_borough, year, month), 2) AS pct_share
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
WHERE pu_borough IS NOT NULL
GROUP BY pu_borough, year, month, service_type
ORDER BY pu_borough, year, month, service_type
"""
query_obt(qj).show(100, truncate=False)

## (k) Top 20 flujos PU→DO por volumen y su ticket promedio

In [ ]:
qk = """
SELECT pu_zone, do_zone,
       COUNT(*) AS trips,
       ROUND(AVG(total_amount), 2) AS avg_ticket,
       ROUND(AVG(trip_distance), 2) AS avg_distance
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
WHERE pu_zone IS NOT NULL AND do_zone IS NOT NULL
GROUP BY pu_zone, do_zone
ORDER BY trips DESC
LIMIT 20
"""
query_obt(qk).show(20, truncate=False)

## (l) Distribución de passenger_count y efecto en total_amount

In [ ]:
ql = """
SELECT passenger_count,
       COUNT(*) AS trips,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_total,
       ROUND(AVG(total_amount), 2) AS avg_total_amount,
       ROUND(AVG(trip_distance), 2) AS avg_distance
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
WHERE passenger_count IS NOT NULL
GROUP BY passenger_count
ORDER BY passenger_count
"""
query_obt(ql).show(truncate=False)

## (m) Impacto de tolls_amount y congestion_surcharge por zona

In [ ]:
qm = """
SELECT pu_zone,
       COUNT(*) AS trips,
       ROUND(AVG(tolls_amount), 2)          AS avg_tolls,
       ROUND(SUM(tolls_amount), 0)           AS total_tolls,
       ROUND(AVG(congestion_surcharge), 2)   AS avg_congestion,
       ROUND(SUM(congestion_surcharge), 0)    AS total_congestion,
       ROUND(AVG(total_amount), 2)           AS avg_total
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
WHERE pu_zone IS NOT NULL
GROUP BY pu_zone
ORDER BY total_congestion DESC
LIMIT 20
"""
query_obt(qm).show(20, truncate=False)

## (n) Proporción de viajes cortos vs largos por borough y estacionalidad

In [ ]:
qn = """
SELECT pu_borough,
       CASE
           WHEN month IN (12, 1, 2)  THEN 'Winter'
           WHEN month IN (3, 4, 5)   THEN 'Spring'
           WHEN month IN (6, 7, 8)   THEN 'Summer'
           WHEN month IN (9, 10, 11) THEN 'Fall'
       END AS season,
       CASE
           WHEN trip_distance <= 2 THEN 'Short (≤2 mi)'
           WHEN trip_distance <= 10 THEN 'Medium (2-10 mi)'
           ELSE 'Long (>10 mi)'
       END AS distance_category,
       COUNT(*) AS trips,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY pu_borough,
           CASE WHEN month IN (12,1,2) THEN 'Winter' WHEN month IN (3,4,5) THEN 'Spring'
                WHEN month IN (6,7,8) THEN 'Summer' ELSE 'Fall' END), 2) AS pct_share
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
WHERE pu_borough IS NOT NULL AND trip_distance IS NOT NULL
GROUP BY pu_borough, season, distance_category
ORDER BY pu_borough, season, distance_category
"""
query_obt(qn).show(100, truncate=False)

## (o) Diferencias por vendor en avg_speed_mph y trip_duration_min

In [ ]:
qo = """
SELECT vendor_name,
       COUNT(*) AS trips,
       ROUND(AVG(avg_speed_mph), 2)     AS avg_speed,
       ROUND(AVG(trip_duration_min), 2)  AS avg_duration,
       ROUND(AVG(trip_distance), 2)      AS avg_distance,
       ROUND(AVG(total_amount), 2)       AS avg_total
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
WHERE vendor_name IS NOT NULL
GROUP BY vendor_name
ORDER BY trips DESC
"""
query_obt(qo).show(truncate=False)

## (p) Relación método de pago ↔ tip_amount por hora

In [ ]:
qp = """
SELECT payment_type_desc, pickup_hour,
       COUNT(*) AS trips,
       ROUND(AVG(tip_amount), 2) AS avg_tip,
       ROUND(AVG(tip_pct), 2)    AS avg_tip_pct
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
WHERE payment_type_desc IS NOT NULL
GROUP BY payment_type_desc, pickup_hour
ORDER BY payment_type_desc, pickup_hour
"""
query_obt(qp).show(100, truncate=False)

## (q) Zonas con percentil 99 de duración/distancia fuera de rango (posible congestión/eventos)

In [ ]:
qq = """
WITH thresholds AS (
    SELECT
        PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY trip_duration_min) AS p99_duration,
        PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY trip_distance)     AS p99_distance
    FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
    WHERE trip_duration_min >= 0 AND trip_distance >= 0
)
SELECT pu_zone, pu_borough,
       COUNT(*) AS extreme_trips,
       ROUND(AVG(trip_duration_min), 2) AS avg_duration,
       ROUND(AVG(trip_distance), 2)     AS avg_distance
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS, thresholds
WHERE (trip_duration_min > p99_duration OR trip_distance > p99_distance)
  AND pu_zone IS NOT NULL
GROUP BY pu_zone, pu_borough
ORDER BY extreme_trips DESC
LIMIT 20
"""
query_obt(qq).show(20, truncate=False)

## (r) Yield por milla (total_amount / trip_distance) por borough y hora

In [ ]:
qr = """
SELECT pu_borough, pickup_hour,
       COUNT(*) AS trips,
       ROUND(AVG(CASE WHEN trip_distance > 0 THEN total_amount / trip_distance END), 2) AS yield_per_mile
FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
WHERE pu_borough IS NOT NULL AND trip_distance > 0 AND total_amount > 0
GROUP BY pu_borough, pickup_hour
ORDER BY pu_borough, pickup_hour
"""
query_obt(qr).show(100, truncate=False)

## (s) Cambios YoY en volumen y ticket promedio por service_type

In [ ]:
qs = """
WITH yearly AS (
    SELECT service_type, year,
           COUNT(*)                    AS trips,
           ROUND(AVG(total_amount), 2) AS avg_ticket
    FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
    GROUP BY service_type, year
)
SELECT y.service_type, y.year,
       y.trips,
       y.avg_ticket,
       ROUND(100.0 * (y.trips - LAG(y.trips) OVER (PARTITION BY y.service_type ORDER BY y.year))
             / NULLIF(LAG(y.trips) OVER (PARTITION BY y.service_type ORDER BY y.year), 0), 2) AS yoy_volume_pct,
       ROUND(y.avg_ticket - LAG(y.avg_ticket) OVER (PARTITION BY y.service_type ORDER BY y.year), 2) AS yoy_ticket_change
FROM yearly y
ORDER BY y.service_type, y.year
"""
query_obt(qs).show(50, truncate=False)

## (t) Días con alta congestion_surcharge: efecto en total_amount vs días "normales"

In [ ]:
qt = """
WITH daily_avg AS (
    SELECT pickup_date,
           AVG(congestion_surcharge) AS avg_daily_congestion,
           AVG(total_amount)         AS avg_daily_total,
           COUNT(*)                  AS trips
    FROM NYC_TAXI_P3.ANALYTICS.OBT_TRIPS
    WHERE pickup_date IS NOT NULL AND congestion_surcharge IS NOT NULL
    GROUP BY pickup_date
),
thresholds AS (
    SELECT PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY avg_daily_congestion) AS p90_congestion
    FROM daily_avg
)
SELECT
    CASE WHEN d.avg_daily_congestion >= t.p90_congestion THEN 'High Congestion Day'
         ELSE 'Normal Day' END                         AS day_type,
    COUNT(*)                                            AS num_days,
    ROUND(AVG(d.avg_daily_total), 2)                    AS avg_total_amount,
    ROUND(AVG(d.avg_daily_congestion), 2)               AS avg_congestion_surcharge,
    ROUND(AVG(d.trips), 0)                              AS avg_daily_trips
FROM daily_avg d, thresholds t
GROUP BY day_type
ORDER BY day_type
"""
query_obt(qt).show(truncate=False)

---
## Resumen

Se respondieron las 20 preguntas de negocio usando SQL pushdown contra `NYC_TAXI_P3.ANALYTICS.OBT_TRIPS` vía Spark + Snowflake connector.